# V49 LIVE — Matched Mixout-Anchored Sarcasm Fine-Tuning

**Protected champion:** V38 = **0.608**  
**Minimum goal:** **0.609**  
**Main target:** **0.620**

MixSarc is excluded from executable V49 because its competition-specific license compatibility remains unresolved. V49 therefore tests the single fallback from the Astra report: **ordinary sarcasm fine-tuning vs the same model with fixed Mixout p=0.7 toward pretrained encoder weights**.

Primary evaluation uses only the 3297 LIVE training rows, with 3 fresh normalized-text group folds (seed 4901), 20% inner calibration groups, and no outer-label threshold selection. The historical 500 validation rows are opened only after the primary gate is frozen.

Promotion gate: mean outer sarcasm F1 gain >= .015; mean official-score gain >= .00125; pooled OOF F1 gain >= .012; wins >=2/3; worst fold F1 >= -.020; mean AP gain >= .005; worst AP >= -.010; bootstrap positive fraction >=90%; bootstrap 10th percentile official gain >0; no eligible script subgroup loss < -.030; zero group leakage. Then the old-500 V38 sarcasm F1/AP acts only as a veto.

This notebook **does not create a Kaggle submission**. Upload `v49_results.zip` to ChatGPT.

> **FIXED build:** JSON export now converts NumPy scalar booleans/numbers to native Python types. If your previous run already completed training, Drive caches will be reused; no full retraining should be necessary.

In [ ]:
# 1. Upload data bundle
from google.colab import files
import pathlib,zipfile
u=files.upload(); assert "v49_LIVE_mixout_sarcasm_data.zip" in u
DATA_DIR=pathlib.Path("/content/v49_data"); DATA_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile("v49_LIVE_mixout_sarcasm_data.zip") as z: z.extractall(DATA_DIR)
for p in sorted(DATA_DIR.iterdir()): print(p.name,f"{p.stat().st_size:,} bytes")

In [ ]:
# 2. Install pinned dependencies
import sys,subprocess
subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1","scikit-learn>=1.4","scipy","safetensors","huggingface_hub","tqdm"])

In [ ]:
# 3. Imports + frozen configuration
import gc,re,time,math,json,random,hashlib,unicodedata,platform
from pathlib import Path
import numpy as np,pandas as pd,torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,Sampler
from tqdm.auto import tqdm
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score,precision_score,recall_score,average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup
from huggingface_hub import snapshot_download
SEED_SPLIT=4901; CALIB_SEEDS=[4911,4912,4913]; TRAIN_SEEDS=[4921,4922,4923]
MODEL_REPO="aplycaebous/tb-BanglaBERT-fpt"; MODEL_REVISION="0b24caaab62d12457ed0eb00ff0f459a1ea509bc"; EXPECTED_WEIGHT_SHA256="9865ebfadc6f38ca385913d64bfa35eadfa03805aeefe42d8465ca075b353a78"
N_OUTER=3; MAX_LENGTH=128; TRAIN_BATCH=8; EVAL_BATCH=64; ACCUM=2; EPOCHS=3; SNAPSHOT_EPOCHS=[2,3]
BACKBONE_LR=1.2e-5; HEAD_LR=2.5e-5; WEIGHT_DECAY=.01; WARMUP_RATIO=.10; MIXOUT_P=.70; GRAD_CLIP=1.0
THRESH_GRID=np.round(np.arange(.05,.951,.05),2); CHECKPOINT_EVERY_OPT_STEPS=25
PROTECTED_PUBLIC_LB=.608; MIN_PUBLIC_LB_GOAL=.609; MAIN_PUBLIC_LB_GOAL=.620
from google.colab import drive
drive.mount('/content/drive'); RUN_DIR=Path('/content/drive/MyDrive/V49_Mixout_Sarcasm'); RUN_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_DIR=Path('/content/v49_export'); EXPORT_DIR.mkdir(exist_ok=True)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
seed_all(TRAIN_SEEDS[0]); assert torch.cuda.is_available(),"Enable T4 GPU"; print(torch.cuda.get_device_name(0))

In [ ]:
# 4. Download exact model revision and verify V48-recorded weight hash
MODEL_DIR=Path('/content/tb_banglabert_fpt_pinned')
if not MODEL_DIR.exists() or not any(MODEL_DIR.iterdir()): snapshot_download(repo_id=MODEL_REPO,revision=MODEL_REVISION,local_dir=str(MODEL_DIR))
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
weights=[p for p in [MODEL_DIR/'pytorch_model.bin',MODEL_DIR/'model.safetensors'] if p.exists()]; assert weights
weight_hashes={p.name:sha256_file(p) for p in weights}; print(weight_hashes)
assert EXPECTED_WEIGHT_SHA256 in weight_hashes.values(),"Pinned weights differ from V48-recorded checkpoint; stop."
print('✅ exact model weights verified')

In [ ]:
# 5. Load and verify exact LIVE partition and hashes
train=pd.read_csv(DATA_DIR/'train.csv',dtype={'ID':str}); valid500=pd.read_csv(DATA_DIR/'validation.csv',dtype={'ID':str}); test=pd.read_csv(DATA_DIR/'test.csv',dtype={'ID':str}); v2v=pd.read_csv(DATA_DIR/'v2_valid.csv',dtype={'ID':str}); v2t=pd.read_csv(DATA_DIR/'v2_test.csv',dtype={'ID':str}); champ=pd.read_csv(DATA_DIR/'submission_v38_champion.csv',dtype={'ID':str})
with open(DATA_DIR/'v49_manifest.json') as f: supplied_manifest=json.load(f)
assert (len(train),len(valid500),len(test))==(3297,500,1000); assert train.ID.is_unique and valid500.ID.is_unique and test.ID.is_unique
assert not(set(train.ID)&set(valid500.ID)) and not(set(train.ID)&set(test.ID)) and not(set(valid500.ID)&set(test.ID))
assert valid500.ID.tolist()==v2v.ID.tolist(); assert test.ID.tolist()==v2t.ID.tolist()==champ.ID.tolist(); assert test.ID.head(4).tolist()==['BICH_0003','BICH_0006','BICH_0015','BICH_0016']
for n in ['train.csv','validation.csv','test.csv','v2_valid.csv','v2_test.csv','submission_v38_champion.csv']:
    assert sha256_file(DATA_DIR/n)==supplied_manifest['files'][n]['sha256'],f'hash mismatch {n}'
print('✅ LIVE IDs and hashes verified')

In [ ]:
# 6. Text views, script categories, fresh group folds
BN_RE=re.compile(r'[\u0980-\u09FF]'); LAT_RE=re.compile(r'[A-Za-z]')
def model_text(x):
    x=unicodedata.normalize('NFKC',str(x)).replace('\u200c',' ').replace('\u200d',' '); return re.sub(r'\s+',' ',x).strip()
def group_key(x): return model_text(x).lower()
def script_type(x):
    s=str(x); bn=bool(BN_RE.search(s)); lat=bool(LAT_RE.search(s)); return 'mixed' if bn and lat else ('native' if bn else 'no_bengali')
for d in [train,valid500,test]: d['model_text']=d.text.map(model_text); d['group_key']=d.text.map(group_key); d['script']=d.model_text.map(script_type)
groups=pd.DataFrame([{'group_key':g,'rows':len(d),'sarcasm_pos':int(d.sarcasm.max()),'script':d.script.mode().iloc[0]} for g,d in train.groupby('group_key',sort=True)])
def strata_for(gdf,min_count):
    s=gdf.sarcasm_pos.astype(str)+'_'+gdf.script.astype(str); c=s.value_counts(); rare=set(c[c<min_count].index)
    if rare: s=pd.Series([str(gdf.sarcasm_pos.iloc[i]) if s.iloc[i] in rare else s.iloc[i] for i in range(len(s))],index=gdf.index)
    return s.astype(str).to_numpy()
skf=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED_SPLIT); groups['outer_fold']=-1
for f,(_,gi) in enumerate(skf.split(groups,strata_for(groups,3))): groups.loc[groups.index[gi],'outer_fold']=f
train['outer_fold']=train.group_key.map(dict(zip(groups.group_key,groups.outer_fold))).astype(int); inner_maps={}; rows=[]
for f in range(3):
    og=groups[groups.outer_fold!=f].copy().reset_index(drop=True); iskf=StratifiedKFold(n_splits=5,shuffle=True,random_state=CALIB_SEEDS[f]); ti,ci=next(iskf.split(og,strata_for(og,5)))
    tg=set(og.iloc[ti].group_key); cg=set(og.iloc[ci].group_key); ogs=set(groups.loc[groups.outer_fold==f,'group_key']); assert tg.isdisjoint(cg) and tg.isdisjoint(ogs) and cg.isdisjoint(ogs)
    inner_maps[f]={'train_groups':tg,'calib_groups':cg,'outer_groups':ogs}
    for role,gs in [('train',tg),('calib',cg),('outer',ogs)]:
        d=train[train.group_key.isin(gs)]; rows.append({'fold':f,'role':role,'rows':len(d),'groups':len(gs),'sarcasm_pos_rows':int(d.sarcasm.sum()),'sarcasm_pos_groups':int(d.groupby('group_key').sarcasm.max().sum())})
split_report=pd.DataFrame(rows); display(split_report); split_report.to_csv(EXPORT_DIR/'v49_split_report.csv',index=False); print('✅ zero group leakage')

In [ ]:
# 7. Dataset + metrics
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_DIR),use_fast=True)
class DS(Dataset):
    def __init__(self,d,labels=True): self.x=d.model_text.astype(str).tolist(); self.y=d.sarcasm.to_numpy(np.float32) if labels else None
    def __len__(self): return len(self.x)
    def __getitem__(self,i):
        q=tokenizer(self.x[i],truncation=True,max_length=MAX_LENGTH,padding=False)
        if self.y is not None: q['labels']=float(self.y[i])
        return q
class Coll:
    def __call__(self,items):
        y=None
        if 'labels' in items[0]: y=torch.tensor([x.pop('labels') for x in items],dtype=torch.float32)
        b=tokenizer.pad(items,padding=True,pad_to_multiple_of=8,return_tensors='pt')
        if y is not None: b['labels']=y
        return b
coll=Coll()
class FixedSampler(Sampler):
    def __init__(self,o): self.o=list(map(int,o))
    def __iter__(self): return iter(self.o)
    def __len__(self): return len(self.o)
def f1p(y,p): return f1_score(y,p,pos_label=1,zero_division=0)
def choose_th(y,p):
    a=[(f1p(y,(p>=t).astype(int)),-abs(float(t)-.5),float(t)) for t in THRESH_GRID]; a.sort(reverse=True); return a[0][2],a[0][0]
def logit_np(p): p=np.clip(np.asarray(p,float),1e-6,1-1e-6); return np.log(p/(1-p))
def margin(p,t): return logit_np(p)-logit_np([t])[0]
@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        b.pop('labels',None); inp={k:v.cuda(non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type='cuda',dtype=torch.float16): z=model(**inp).logits.squeeze(-1)
        out.append(torch.sigmoid(z.float()).cpu().numpy())
    return np.concatenate(out)
def eval_loader(d,labels=True): return DataLoader(DS(d.reset_index(drop=True),labels),batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,collate_fn=coll)

In [ ]:
# 8. Mixout implementation + integrity tests
MIXGEN=None
class MixLinear(nn.Module):
    def __init__(self,base,p):
        super().__init__(); self.in_features=base.in_features; self.out_features=base.out_features; self.p=float(p); self.weight=nn.Parameter(base.weight.detach().clone()); self.bias=None if base.bias is None else nn.Parameter(base.bias.detach().clone()); self.register_buffer('anchor_weight',base.weight.detach().clone())
    def forward(self,x):
        global MIXGEN
        if (not self.training) or self.p<=0: w=self.weight
        else:
            keep=1-self.p; m=(torch.rand((1,self.in_features),device=self.weight.device,generator=MIXGEN)<keep).to(self.weight.dtype)/keep; w=self.anchor_weight+(self.weight-self.anchor_weight)*m
        return F.linear(x,w,self.bias)
def wrap_encoder(model,p):
    assert hasattr(model,'electra') and hasattr(model.electra,'encoder'); names=[]
    def rec(mod,prefix):
        for n,ch in list(mod.named_children()):
            full=f'{prefix}.{n}'
            if isinstance(ch,nn.Linear): setattr(mod,n,MixLinear(ch,p)); names.append(full)
            else: rec(ch,full)
    rec(model.electra.encoder,'electra.encoder'); assert names; return names
def build_model(seed,arm):
    global MIXGEN
    seed_all(seed); m=AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR),num_labels=1,problem_type='multi_label_classification',ignore_mismatched_sizes=True); names=[]
    if arm=='mixout': MIXGEN=torch.Generator(device='cuda'); MIXGEN.manual_seed(seed+70000); names=wrap_encoder(m,MIXOUT_P)
    else: MIXGEN=None
    return m.cuda(),names
def optimizer_for(m):
    head=[]; back=[]
    for n,p in m.named_parameters(): (head if ('classifier' in n or 'score' in n) else back).append(p)
    return torch.optim.AdamW([{'params':back,'lr':BACKBONE_LR,'weight_decay':WEIGHT_DECAY},{'params':head,'lr':HEAD_LR,'weight_decay':WEIGHT_DECAY}])
# p=0 equivalence
seed_all(123); b=nn.Linear(13,7); w=MixLinear(b,0.0); x=torch.randn(5,13); b.eval(); w.eval(); assert torch.equal(b(x),w(x)); w.train(); w(x).sum().backward(); assert w.anchor_weight.grad is None
tmp,names=build_model(TRAIN_SEEDS[0],'mixout'); assert all(n.startswith('electra.encoder.') for n in names) and not any('classifier' in n for n in names); t=tokenizer(['এটা পরীক্ষা','test'],padding=True,return_tensors='pt'); t={k:v.cuda() for k,v in t.items()}; tmp.eval();
with torch.no_grad(): a=tmp(**t).logits; bb=tmp(**t).logits
assert torch.equal(a,bb); print('✅ Mixout tests passed; wrapped',len(names),'encoder Linear layers'); del tmp; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 9. Checkpoint/resume helpers
FP=hashlib.sha256(json.dumps({'rev':MODEL_REVISION,'train_sha':supplied_manifest['files']['train.csv']['sha256'],'split':SEED_SPLIT,'calib':CALIB_SEEDS,'train_seeds':TRAIN_SEEDS,'mixout_p':MIXOUT_P,'epochs':EPOCHS,'lr':[BACKBONE_LR,HEAD_LR]},sort_keys=True).encode()).hexdigest()
def rng_state():
    s={'python':random.getstate(),'numpy':np.random.get_state(),'cpu':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()};
    if MIXGEN is not None: s['mix']=MIXGEN.get_state()
    return s
def restore_rng(s):
    global MIXGEN
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['cpu']); torch.cuda.set_rng_state_all(s['cuda']);
    if MIXGEN is not None and 'mix' in s: MIXGEN.set_state(s['mix'])
def save_ck(p,m,o,sc,gs,e,nb,os): torch.save({'fp':FP,'model':m.state_dict(),'opt':o.state_dict(),'sched':sc.state_dict(),'scaler':gs.state_dict(),'epoch':e,'next_batch':nb,'opt_steps':os,'rng':rng_state()},p)
def load_ck(p,m,o,sc,gs):
    c=torch.load(p,map_location='cpu'); assert c['fp']==FP; m.load_state_dict(c['model']); o.load_state_dict(c['opt']); sc.load_state_dict(c['sched']); gs.load_state_dict(c['scaler']); restore_rng(c['rng']); return c['epoch'],c['next_batch'],c['opt_steps']

In [ ]:
# 10. Throughput pilot (no outcomes examined)
pilot=train[train.group_key.isin(inner_maps[0]['train_groups'])].reset_index(drop=True); times={}
for arm in ['control','mixout']:
    m,_=build_model(TRAIN_SEEDS[0],arm); o=optimizer_for(m); gs=torch.amp.GradScaler('cuda'); order=np.random.default_rng(TRAIN_SEEDS[0]).permutation(len(pilot)); ld=DataLoader(DS(pilot,True),batch_size=TRAIN_BATCH,sampler=FixedSampler(order),num_workers=0,pin_memory=True,collate_fn=coll); m.train(); o.zero_grad(set_to_none=True); torch.cuda.synchronize(); t0=time.time(); seen=0
    for i,b in enumerate(ld):
        if i>=10: break
        y=b.pop('labels').cuda(); inp={k:v.cuda(non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type='cuda',dtype=torch.float16): loss=F.binary_cross_entropy_with_logits(m(**inp).logits.squeeze(-1).float(),y)
        gs.scale(loss).backward(); o.zero_grad(set_to_none=True); seen+=1
    torch.cuda.synchronize(); times[arm]=(time.time()-t0)/seen; del m,o,gs,ld; gc.collect(); torch.cuda.empty_cache()
sec=0
for f in range(3):
    n=len(train[train.group_key.isin(inner_maps[f]['train_groups'])]); batches=math.ceil(n/TRAIN_BATCH); sec+=batches*EPOCHS*(times['control']+times['mixout'])
project=sec/60*1.25+30; print('sec/batch',times,'projected budget min',project); assert project<=300,'Projected runtime exceeds 5h; stop without reducing design.' 

In [ ]:
# 11. Train one arm with mid-epoch resume
def train_arm(fold,arm):
    global MIXGEN
    seed=TRAIN_SEEDS[fold]; mm=inner_maps[fold]; tr=train[train.group_key.isin(mm['train_groups'])].copy().reset_index(drop=True); ca=train[train.group_key.isin(mm['calib_groups'])].copy().reset_index(drop=True); ou=train[train.group_key.isin(mm['outer_groups'])].copy().reset_index(drop=True)
    complete=RUN_DIR/f'fold{fold}_{arm}_complete.npz'; ck=RUN_DIR/f'fold{fold}_{arm}_latest.pt'; hp=RUN_DIR/f'fold{fold}_{arm}_history.csv'
    if complete.exists():
        print('✅ reuse',complete.name); z=np.load(complete); return {k:z[k] for k in z.files}
    m,names=build_model(seed,arm); o=optimizer_for(m); bpe=math.ceil(len(tr)/TRAIN_BATCH); total=math.ceil(bpe/ACCUM)*EPOCHS; sc=get_linear_schedule_with_warmup(o,int(total*WARMUP_RATIO),total); gs=torch.amp.GradScaler('cuda'); se,sb,ops=1,0,0
    if ck.exists(): se,sb,ops=load_ck(ck,m,o,sc,gs); print('↩ resume',fold,arm,se,sb)
    cal=eval_loader(ca,True); out=eval_loader(ou,True); vld=eval_loader(valid500,True); tel=eval_loader(test,False); hist=pd.read_csv(hp).to_dict('records') if hp.exists() else []; t0=time.time()
    for e in range(se,EPOCHS+1):
        es=seed+e*101; resumed=(e==se and sb>0 and ck.exists())
        if not resumed:
            seed_all(es)
            if arm=='mixout': MIXGEN.manual_seed(es+70000)
        order=np.random.default_rng(es).permutation(len(tr)); ld=DataLoader(DS(tr,True),batch_size=TRAIN_BATCH,sampler=FixedSampler(order),num_workers=0,pin_memory=True,collate_fn=coll); m.train(); o.zero_grad(set_to_none=True); start=sb if e==se else 0; running=0; seen=0
        for step,b in enumerate(tqdm(ld,desc=f'f{fold} {arm} e{e}/3')):
            if step<start: continue
            y=b.pop('labels').cuda(non_blocking=True); inp={k:v.cuda(non_blocking=True) for k,v in b.items()}
            with torch.autocast(device_type='cuda',dtype=torch.float16): loss=F.binary_cross_entropy_with_logits(m(**inp).logits.squeeze(-1).float(),y)
            gs.scale(loss/ACCUM).backward(); running+=float(loss.item())*len(y); seen+=len(y); do=((step+1)%ACCUM==0 or step+1==len(ld))
            if do:
                gs.unscale_(o); torch.nn.utils.clip_grad_norm_(m.parameters(),GRAD_CLIP); old=gs.get_scale(); gs.step(o); gs.update();
                if gs.get_scale()>=old: sc.step()
                o.zero_grad(set_to_none=True); ops+=1
                if ops%CHECKPOINT_EVERY_OPT_STEPS==0: save_ck(ck,m,o,sc,gs,e,step+1,ops)
        if e in SNAPSHOT_EPOCHS:
            sp=RUN_DIR/f'fold{fold}_{arm}_epoch{e}_preds.npz'
            if not sp.exists(): np.savez_compressed(sp,calib_prob=predict(m,cal).astype(np.float32),outer_prob=predict(m,out).astype(np.float32),valid500_prob=predict(m,vld).astype(np.float32),test_prob=predict(m,tel).astype(np.float32))
        hist.append({'fold':fold,'arm':arm,'epoch':e,'mean_loss_since_resume':running/max(seen,1),'opt_steps':ops,'elapsed_min':(time.time()-t0)/60,'wrapped_linears':len(names)}); pd.DataFrame(hist).to_csv(hp,index=False); save_ck(ck,m,o,sc,gs,e+1,0,ops); sb=0
    snaps=[np.load(RUN_DIR/f'fold{fold}_{arm}_epoch{e}_preds.npz') for e in SNAPSHOT_EPOCHS]; ret={'calib_ids':ca.ID.astype(str).to_numpy(),'outer_ids':ou.ID.astype(str).to_numpy(),'calib_y':ca.sarcasm.to_numpy(np.int8),'outer_y':ou.sarcasm.to_numpy(np.int8),'calib_prob':np.mean([z['calib_prob'] for z in snaps],0).astype(np.float32),'outer_prob':np.mean([z['outer_prob'] for z in snaps],0).astype(np.float32),'valid500_prob':np.mean([z['valid500_prob'] for z in snaps],0).astype(np.float32),'test_prob':np.mean([z['test_prob'] for z in snaps],0).astype(np.float32)}; np.savez_compressed(complete,**ret); del m,o,sc,gs,cal,out,vld,tel; gc.collect(); torch.cuda.empty_cache(); return ret

In [ ]:
# 12. Execute six fits + fold metrics
raw={}; fr=[]; start=time.time()
for f in range(3):
    raw[f]={a:train_arm(f,a) for a in ['control','mixout']}; c=raw[f]['control']; m=raw[f]['mixout']; assert c['outer_ids'].tolist()==m['outer_ids'].tolist() and c['calib_ids'].tolist()==m['calib_ids'].tolist(); tc,_=choose_th(c['calib_y'],c['calib_prob']); tm,_=choose_th(m['calib_y'],m['calib_prob']); y=c['outer_y'].astype(int); pc=(c['outer_prob']>=tc).astype(int); pm=(m['outer_prob']>=tm).astype(int); fc=f1p(y,pc); fm=f1p(y,pm); ac=average_precision_score(y,c['outer_prob']); am=average_precision_score(y,m['outer_prob']); fr.append({'fold':f,'control_threshold':tc,'mixout_threshold':tm,'control_f1':fc,'mixout_f1':fm,'f1_gain':fm-fc,'official_gain':(fm-fc)/12,'control_ap':ac,'mixout_ap':am,'ap_gain':am-ac,'control_precision':precision_score(y,pc,zero_division=0),'mixout_precision':precision_score(y,pm,zero_division=0),'control_recall':recall_score(y,pc,zero_division=0),'mixout_recall':recall_score(y,pm,zero_division=0),'true_support':int(y.sum()),'control_pred_pos':int(pc.sum()),'mixout_pred_pos':int(pm.sum())})
fold_report=pd.DataFrame(fr); display(fold_report.round(6)); fold_report.to_csv(EXPORT_DIR/'v49_fold_results.csv',index=False); print('elapsed min',(time.time()-start)/60)

In [ ]:
# 13. Pooled OOF + script diagnostics + 5,000 group bootstrap
co=np.full(len(train),np.nan,np.float32); mo=co.copy(); cp=np.full(len(train),-1,np.int8); mp=cp.copy(); fc=np.full(len(train),-1,np.int8); id2i={x:i for i,x in enumerate(train.ID.astype(str))}; thc=[]; thm=[]
for f in range(3):
    c=raw[f]['control']; m=raw[f]['mixout']; tc,_=choose_th(c['calib_y'],c['calib_prob']); tm,_=choose_th(m['calib_y'],m['calib_prob']); thc.append(tc); thm.append(tm); idx=np.array([id2i[x] for x in c['outer_ids'].astype(str)]); co[idx]=c['outer_prob']; mo[idx]=m['outer_prob']; cp[idx]=(c['outer_prob']>=tc); mp[idx]=(m['outer_prob']>=tm); fc[idx]=f
assert np.isfinite(co).all() and np.isfinite(mo).all(); y=train.sarcasm.to_numpy(int); pooled={'control_f1':f1p(y,cp),'mixout_f1':f1p(y,mp),'f1_gain':f1p(y,mp)-f1p(y,cp),'official_gain':(f1p(y,mp)-f1p(y,cp))/12,'control_ap':average_precision_score(y,co),'mixout_ap':average_precision_score(y,mo),'ap_gain':average_precision_score(y,mo)-average_precision_score(y,co)}; print('pooled',pooled)
oof=pd.DataFrame({'ID':train.ID.astype(str),'outer_fold':fc,'group_key':train.group_key,'script':train.script,'y':y,'control_prob':co,'mixout_prob':mo,'control_pred':cp,'mixout_pred':mp}); oof.to_csv(EXPORT_DIR/'v49_oof_predictions.csv',index=False)
sr=[]
for s in ['native','mixed','no_bengali']:
    mask=train.script.to_numpy()==s; pg=int(train.loc[mask].groupby('group_key').sarcasm.max().sum()); a=f1p(y[mask],cp[mask]); b=f1p(y[mask],mp[mask]); sr.append({'script':s,'rows':int(mask.sum()),'positive_groups':pg,'control_f1':a,'mixout_f1':b,'f1_gain':b-a,'gate_eligible':pg>=20})
script_report=pd.DataFrame(sr); display(script_report.round(6)); script_report.to_csv(EXPORT_DIR/'v49_script_subgroups.csv',index=False)
fg={}
for f in range(3): fg[f]={g:np.asarray(list(ix),int) for g,ix in train[train.outer_fold==f].groupby('group_key').groups.items()}
rng=np.random.default_rng(4999); bo=[]
for _ in range(5000):
    idx=[]
    for f in range(3):
        gs=np.array(list(fg[f]),dtype=object); samp=rng.choice(gs,size=len(gs),replace=True); idx.extend(np.concatenate([fg[f][g] for g in samp]).tolist())
    idx=np.asarray(idx,int); bo.append((f1p(y[idx],mp[idx])-f1p(y[idx],cp[idx]))/12)
bo=np.asarray(bo); bootstrap={'n':5000,'mean_official_gain':float(bo.mean()),'p10':float(np.quantile(bo,.10)),'p20':float(np.quantile(bo,.20)),'p2_5':float(np.quantile(bo,.025)),'p97_5':float(np.quantile(bo,.975)),'positive_fraction':float((bo>0).mean())}; print('bootstrap',bootstrap)

In [ ]:
# 14. Freeze primary gate BEFORE opening old-500 labels
mean_f=float(fold_report.f1_gain.mean()); mean_o=float(fold_report.official_gain.mean()); pooled_f=float(pooled['f1_gain']); wins=int((fold_report.f1_gain>0).sum()); worst_f=float(fold_report.f1_gain.min()); mean_ap=float(fold_report.ap_gain.mean()); worst_ap=float(fold_report.ap_gain.min())
script_ok=True; sdet={}
for _,r in script_report.iterrows():
    eligible=bool(r.gate_eligible); ok=(float(r.f1_gain)>=-.03) if eligible else None; sdet[r.script]={'eligible':eligible,'positive_groups':int(r.positive_groups),'f1_gain':float(r.f1_gain),'passes_if_eligible':ok}; script_ok &= (True if not eligible else bool(ok))
criteria={'mean_outer_f1_gain_ge_.015':mean_f>=.015,'mean_official_gain_ge_.00125':mean_o>=.00125,'pooled_oof_f1_gain_ge_.012':pooled_f>=.012,'wins_ge_2_of_3':wins>=2,'worst_fold_f1_ge_minus_.020':worst_f>=-.020,'mean_ap_gain_ge_.005':mean_ap>=.005,'worst_ap_gain_ge_minus_.010':worst_ap>=-.010,'bootstrap_positive_ge_.90':bootstrap['positive_fraction']>=.90,'bootstrap_p10_gt_0':bootstrap['p10']>0,'script_stability':script_ok,'zero_group_leakage':True,'all_six_complete':all((RUN_DIR/f'fold{f}_{a}_complete.npz').exists() for f in range(3) for a in ['control','mixout'])}; PRIMARY_PASS=all(criteria.values()); primary={'mean_outer_f1_gain':mean_f,'mean_official_gain':mean_o,'pooled_oof_f1_gain':pooled_f,'fold_wins':wins,'worst_fold_f1_gain':worst_f,'mean_ap_gain':mean_ap,'worst_ap_gain':worst_ap,'script':sdet,'bootstrap':bootstrap,'criteria':criteria,'PRIMARY_PASS':PRIMARY_PASS}; print(json.dumps(primary,indent=2)); (EXPORT_DIR/'v49_primary_summary.json').write_text(json.dumps(primary,indent=2))

In [ ]:
# 15. Secondary old-500 veto + save test probabilities (no submission)
cm500=[]; mm500=[]; cmtest=[]; mmtest=[]
for f in range(3):
    cm500.append(margin(raw[f]['control']['valid500_prob'],thc[f])); mm500.append(margin(raw[f]['mixout']['valid500_prob'],thm[f])); cmtest.append(margin(raw[f]['control']['test_prob'],thc[f])); mmtest.append(margin(raw[f]['mixout']['test_prob'],thm[f]))
cm500=np.mean(cm500,0); mm500=np.mean(mm500,0); cmtest=np.mean(cmtest,0); mmtest=np.mean(mmtest,0); y5=valid500.sarcasm.to_numpy(int); v38s=v2v.sarcasm_prob.to_numpy(float); v38p=(v38s>=.175).astype(int); c5=(cm500>=0).astype(int); m5=(mm500>=0).astype(int)
secondary={'v38_f1':f1p(y5,v38p),'v49_control_f1':f1p(y5,c5),'v49_mixout_f1':f1p(y5,m5),'v38_ap':average_precision_score(y5,v38s),'v49_control_ap':average_precision_score(y5,cm500),'v49_mixout_ap':average_precision_score(y5,mm500)}; secondary['veto_f1_non_degradation']=bool(secondary['v49_mixout_f1']>=secondary['v38_f1']); secondary['veto_ap_non_degradation']=bool(secondary['v49_mixout_ap']>=secondary['v38_ap']); VETO_PASS=bool(secondary['veto_f1_non_degradation'] and secondary['veto_ap_non_degradation']); ADVANCE=bool(PRIMARY_PASS and VETO_PASS); print('secondary veto',json.dumps(secondary,indent=2)); print('ADVANCE',ADVANCE)
pd.DataFrame({'ID':valid500.ID.astype(str),'v38_v2_sarcasm_prob':v38s,'v49_control_margin':cm500,'v49_mixout_margin':mm500}).to_csv(EXPORT_DIR/'v49_validation500_probabilities.csv',index=False)
pd.DataFrame({'ID':test.ID.astype(str),'v2_sarcasm_prob':v2t.sarcasm_prob.to_numpy(float),'v49_control_margin':cmtest,'v49_mixout_margin':mmtest}).to_csv(EXPORT_DIR/'v49_test_probabilities.csv',index=False)
champ.to_csv(EXPORT_DIR/'submission_v38_PROTECTED.csv',index=False)
import transformers

def _json_safe(obj):
    import numpy as _np
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, _np.ndarray):
        return obj.tolist()
    if isinstance(obj, _np.generic):
        return obj.item()
    return obj

summary={'protected_public_lb':.608,'minimum_public_lb_goal':.609,'main_public_lb_goal':.620,'model_repo':MODEL_REPO,'model_revision':MODEL_REVISION,'mixout_p':MIXOUT_P,'primary':primary,'secondary_old500_veto':secondary,'PRIMARY_PASS':PRIMARY_PASS,'VETO_PASS':bool(VETO_PASS),'ADVANCE':bool(ADVANCE),'decision':'ADVANCE to separate V38 sarcasm integration check; not yet champion.' if ADVANCE else 'REJECT V49; retain V38 = 0.608.','environment':{'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__,'sklearn':sklearn.__version__,'gpu':torch.cuda.get_device_name(0)},'data_manifest':supplied_manifest}; summary=_json_safe(summary); (EXPORT_DIR/'v49_summary.json').write_text(json.dumps(summary,indent=2)); (EXPORT_DIR/'v49_config.json').write_text(json.dumps({'split_seed':SEED_SPLIT,'calib_seeds':CALIB_SEEDS,'train_seeds':TRAIN_SEEDS,'model_repo':MODEL_REPO,'model_revision':MODEL_REVISION,'weight_sha256':EXPECTED_WEIGHT_SHA256,'mixout_p':MIXOUT_P,'epochs':EPOCHS,'batch':TRAIN_BATCH,'accum':ACCUM,'lr':[BACKBONE_LR,HEAD_LR],'threshold_grid':THRESH_GRID.tolist(),'fingerprint':FP},indent=2)); (EXPORT_DIR/'README_DECISION.txt').write_text(f'V49 Mixout sarcasm\nProtected V38=0.608\nMinimum goal=0.609\nPRIMARY_PASS={PRIMARY_PASS}\nVETO_PASS={VETO_PASS}\nADVANCE={ADVANCE}\n{summary["decision"]}\nNo Kaggle challenger submission was generated.')
zip_path=Path('/content/v49_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file(): z.write(p,p.name)
print('Created',zip_path); from google.colab import files; files.download(str(zip_path))